# Extended simulator: mid-training hooks

Extends `simulator.ipynb` for approaches that act **during** a client's local training, not only before and after it. Example: FedASMU, where a client downloads the fresh global model halfway through its training and mixes it into its half-trained model.

**How it's used:** an approach notebook that needs this starts with

```python
%run ../extended_simulator.ipynb
```

That gives it everything from `simulator.ipynb` (settings, data, devices, models, the event loop, `run()`, saving), plus the hook below. Results are saved exactly like the other approaches, so `compare.ipynb` and `run_all.ipynb` include them automatically.

> **Important:** this notebook does **not** change `simulator.ipynb`. It loads it and adds to it. Approaches that use `simulator.ipynb` directly are not affected, and an approach that does not define the hook gives identical results with either simulator.

| Section | What it does |
|---|---|
| 1. Load the shared simulator | everything from `simulator.ipynb` |
| **2. Mid-training hook** | **`on_local_epoch`: called after every local epoch except the last** |

## 1. Load the shared simulator

The code cells of `simulator.ipynb` are run **in this notebook's namespace** (not with a nested `%run`), so that `run()` builds the extended clients and server defined below.

> **Note:** `simulator.ipynb` is found in this folder or a parent folder, so approach notebooks in `approaches/` work too.

In [ ]:
from pathlib import Path
import nbformat

_folder = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulator.ipynb").exists())
for _cell in nbformat.read(_folder / "simulator.ipynb", as_version=4).cells:
    if _cell.cell_type == "code":
        exec(_cell.source)

## 2. Mid-training hook

If an approach defines

```python
def on_local_epoch(self, server, client):
    ...
```

it is called after every local epoch of a client **except the last** one, with the client's model in its current, half-trained state. Inside it the approach can:

| | |
|---|---|
| `client.model` | the client's model right now (change its weights with `client.model.load_state_dict(...)`) |
| `client.epoch` | how many local epochs are finished |
| `client.version` | the global version the client started from |
| `client.pause(seconds)` | stop the client for that many virtual seconds (for example to download a model) |
| `client.batch_gradient()` | the gradient of the client's loss on its next mini-batch, without training on it |

> **Note:** `client.pause` makes the extra communication cost simulated time: the client trains nothing while it waits. The pause starts at the end of the current time slice.

In [ ]:
class ExtendedClient(Client):
    def begin(self, *args, **kwargs):
        super().begin(*args, **kwargs)
        self.pending_pause = 0.0

    def pause(self, seconds):
        self.pending_pause += seconds

    def batch_gradient(self):
        """Gradient of the loss on the next mini-batch (no training step), as a dict like the model's parameters."""
        ids = torch.as_tensor(self.order[self.cursor:self.cursor + self.S.batch_size], dtype=torch.long)
        x, y = self.data.train_x[ids].to(self.device), self.data.train_y[ids].to(self.device)
        self.model.zero_grad()
        F.cross_entropy(self.model(x), y).backward()
        grad = {n: p.grad.detach().clone() for n, p in self.model.named_parameters()}
        self.model.zero_grad()
        return grad

    def _train_batch(self):
        epoch = self.epoch
        done = super()._train_batch()
        hook = getattr(self.server.approach, "on_local_epoch", None)
        if not done and self.epoch > epoch and hook is not None:
            hook(self.server, self)
        return done

    def train_for(self, start, quantum):
        """As Client.train_for, but a requested pause stops training for the rest of the slice and the pause time."""
        if self.finished or start < self.ready_time:
            return 0
        cost = self.S.batch_size / self.S.speed_batch_size if self.S.speed_batch_size else 1.0
        self.credit += self.speed * quantum
        n = 0
        while self.credit + 1e-9 >= cost and not self.finished:
            done = self._train_batch()
            self.credit -= cost
            n += 1
            if done:
                self.finished = True
                self.finish_time = start + quantum
                self.arrival_time = self.finish_time + self.upload_latency
            elif self.pending_pause > 0:
                self.ready_time = start + quantum + self.pending_pause
                self.pending_pause, self.credit = 0.0, 0.0
                break
        return n


class ExtendedServer(Server):
    def _begin(self, c):
        c.server = self                            # lets the client call the approach's hook
        super()._begin(c)


Client, Server = ExtendedClient, ExtendedServer    # run() builds these